In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[1]
sys.path.append(str(project_root))

In [2]:
from src.syfu.api.models.task import Task, TaskPriority
from langchain_nvidia import NVIDIAEmbeddings
from src.syfu.api.models.base import Base
from langchain_nvidia import ChatNVIDIA
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_chroma import Chroma
from sqlalchemy.orm import Session
from src.syfu.api.models import db
from typing import Optional, List
from dotenv import load_dotenv
from datetime import datetime
from sqlalchemy import select
from pprint import pprint
import os

Base.metadata.create_all(db)
load_dotenv()

2026-08-14 01:47:19,182 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-08-14 01:47:19,183 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("task")
2026-08-14 01:47:19,183 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-08-14 01:47:19,185 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("youtube")
2026-08-14 01:47:19,185 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-08-14 01:47:19,186 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("interest")
2026-08-14 01:47:19,186 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-08-14 01:47:19,187 INFO sqlalchemy.engine.Engine COMMIT


True

### Properties Initialization
Here we add the files into their variables that defines what the chatbot needs to do and it's general behavior.

In [3]:
soul = open("./info/soul.md", 'r').read()
pprint(soul)

('# SOUL\n'
 'This is the description of your personality\n'
 '- Your name is Sybot.\n'
 '- You are a helpful assisstant for a productivity application named "SYFU".\n'
 '- You are to be absolutely clinical with your responses.')


### Initialize LLM and Embeddings model

In [4]:
# llm = ChatOpenAI(
#     model="big-pickle",  # or 'deepseek-v4-pro', 'glm-5.2', 'zen-default', etc.
#     openai_api_key=os.getenv("OPENCODE_API_KEY"),
#     openai_api_base="https://opencode.ai/zen/v1",
#     temperature=0,
# )
llm = ChatNVIDIA(
    model="z-ai/glm-5.2",
    temperature=0,
)
embedding = NVIDIAEmbeddings(
  model="nvidia/nemotron-3-embed-1b"
)

pprint(llm.invoke(f"{soul}\nhelllo").content)

/home/big/Projects/syfu/backend/.venv/lib/python3.14/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found z-ai/glm-5.2 in available_models, but type is unknown and inference may fail.
  warnings.warn(
/home/big/Projects/syfu/backend/.venv/lib/python3.14/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


'Greetings. State your SYFU-related query or task.'


### Tools

#### CRUD Tools
For Tasks, Projects, and TimeTable

In [5]:
@tool()
def get_tasks() -> List[Task]:
    """This function is used to get all the tasks.

    Returns:
        List[Task]: A list of all the tasks
    """

    with Session(db) as session:
        stmt = select(Task)
        return session.scalars(stmt)

@tool()
def create_tasks(title: str, 
                description: Optional[str] = None, 
                assoc_date: Optional[datetime] = None, 
                deadline: Optional[datetime] = None, 
                priority: TaskPriority = TaskPriority.CHL) -> Task:
    """This is the function which is used to create a new task.

    Args:
        title (str): _description_
        description (Optional[str], optional): _description_. Defaults to None.
        assoc_date (Optional[datetime], optional): _description_. Defaults to None.
        deadline (Optional[datetime], optional): _description_. Defaults to None.
        priority (TaskPriority, optional): _description_. Defaults to TaskPriority.CHL.

    Returns:
        Task: The task that was just now created is returned
    """
    with Session(db) as session:
        new_task = Task(
            title=title,
            description=description,
            assocDate=assoc_date,
            deadline=deadline,
            priority=priority
        )
        session.add(new_task)
        session.commit()
        session.refresh(new_task)
        return new_task

#### Binding Tools

In [6]:
tools = [create_tasks, get_tasks]
tos = {t.name: t for t in tools}

llm = llm.bind_tools(tools)

/home/big/Projects/syfu/backend/.venv/lib/python3.14/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1044: UserWarning: Model 'z-ai/glm-5.2' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
